In [9]:
from pathlib import Path
import pandas as pd
import numpy as np


# =========================
# Config
# =========================
ROOT = Path(r"D:\mmwave-heart-rate-monitoring-demo\results\filter_v2")

FILES = [
    "AWR_steady.csv",
    "AWR_unsteady.csv",
    "IWR_steady.csv",
    "IWR_unsteady.csv",
]

METHOD_ORDER = [
    "raw",
    "ABP",
    "EMD",
    "ROS",
    "LEN",
    "ABP→EMD",
    "ABP→ROS",
    "ABP→LEN",
    "EMD→ABP",
    "EMD→ROS",
    "EMD→LEN",
    "ROS→ABP",
    "ROS→EMD",
    "ROS→LEN",
    "LEN→ABP",
    "LEN→EMD",
    "LEN→ROS",
]

METHOD_TO_COLUMN = {
    "raw": "raw_HR",
    "ABP": "ABP_HR",
    "EMD": "EMD_HR",
    "ROS": "ROS_HR",
    "LEN": "LEN_HR",
    "ABP→EMD": "ABP→EMD_HR",
    "ABP→ROS": "ABP→ROS_HR",
    "ABP→LEN": "ABP→LEN_HR",
    "EMD→ABP": "EMD→ABP_HR",
    "EMD→ROS": "EMD→ROS_HR",
    "EMD→LEN": "EMD→LEN_HR",
    "ROS→ABP": "ROS→ABP_HR",
    "ROS→EMD": "ROS→EMD_HR",
    "ROS→LEN": "ROS→LEN_HR",
    "LEN→ABP": "LEN→ABP_HR",
    "LEN→EMD": "LEN→EMD_HR",
    "LEN→ROS": "LEN→ROS_HR",
}


# =========================
# Metrics
# =========================
def compute_metrics(ecg, mmw):
    error = mmw - ecg
    mae = np.mean(np.abs(error))
    rmse = np.sqrt(np.mean(error**2))
    bias = np.mean(error)
    std = np.std(error, ddof=1) if len(error) > 1 else 0.0
    ci = 1.96 * std
    return mae, rmse, bias, std, ci


# =========================
# Analysis
# =========================
for file in FILES:
    path = ROOT / file

    if not path.exists():
        print(f"找不到檔案: {path}")
        continue

    df = pd.read_csv(path)

    # 保險：把欄位前後空白去掉
    df.columns = df.columns.str.strip()

    if "ECG_HR" not in df.columns:
        print(f"{file} 缺少 ECG_HR 欄位")
        continue

    ecg = pd.to_numeric(df["ECG_HR"], errors="coerce")
    rows = []

    for method in METHOD_ORDER:
        col = METHOD_TO_COLUMN[method]

        if col not in df.columns:
            print(f"找不到欄位: {col}")
            rows.append({
                "Method": method,
                "MAE": np.nan,
                "RMSE": np.nan,
                "Bias": np.nan,
                "Std": np.nan,
                "95% CI": np.nan
            })
            continue

        mmw = pd.to_numeric(df[col], errors="coerce")
        valid = np.isfinite(mmw) & np.isfinite(ecg)

        if valid.sum() == 0:
            rows.append({
                "Method": method,
                "MAE": np.nan,
                "RMSE": np.nan,
                "Bias": np.nan,
                "Std": np.nan,
                "95% CI": np.nan
            })
            continue

        mae, rmse, bias, std, ci = compute_metrics(
            ecg[valid].to_numpy(),
            mmw[valid].to_numpy()
        )

        rows.append({
            "Method": method,
            "MAE": round(mae, 3),
            "RMSE": round(rmse, 3),
            "Bias": round(bias, 3),
            "Std": round(std, 3),
            "95% CI": round(ci, 3)
        })

    result = pd.DataFrame(rows)

    print("\n" + "=" * 60)
    print(file)
    print("=" * 60)
    print(result.to_string(index=False))

    # out_path = ROOT / f"{Path(file).stem}_stats.csv"
    # result.to_csv(out_path, index=False, encoding="utf-8-sig")
    # print(f"\n已儲存: {out_path}")


AWR_steady.csv
 Method    MAE   RMSE   Bias    Std  95% CI
    raw  5.279  7.377  2.492  7.014  13.747
    ABP  7.334 12.987  5.211 12.017  23.553
    EMD 17.703 28.685 16.579 23.646  46.346
    ROS 10.112 20.276  8.457 18.615  36.486
    LEN  4.899  6.486  2.117  6.193  12.138
ABP→EMD  7.557 13.231  5.586 12.115  23.746
ABP→ROS  7.399 13.069  5.375 12.034  23.586
ABP→LEN  6.093 11.233  3.277 10.853  21.272
EMD→ABP 17.774 28.869 16.697 23.790  46.628
EMD→ROS 19.360 32.125 18.267 26.694  52.320
EMD→LEN 11.416 20.856  9.349 18.832  36.911
ROS→ABP 12.489 23.576 11.246 20.931  41.024
ROS→EMD 20.793 32.748 19.863 26.301  51.550
ROS→LEN  4.632  6.113  1.531  5.978  11.716
LEN→ABP  5.204  6.985  2.398  6.627  12.988
LEN→EMD 11.188 21.278  9.875 19.040  37.317
LEN→ROS  6.222 10.655  3.324 10.226  20.043

AWR_unsteady.csv
 Method    MAE   RMSE    Bias    Std  95% CI
    raw 26.605 30.717 -21.310 22.262  43.634
    ABP 19.616 24.252 -12.959 20.628  40.431
    EMD 14.153 19.322  -1.302 19.400  3